# Módulo 3 - Ejercicio 4 (Plantilla)
## Ejecución del Algoritmo de Grover en IBM Quantum y Análisis Experimental

### 1. Contexto
El Algoritmo de Grover nos permite encontrar un elemento marcado dentro de un espacio no estructurado de $N = 2^n$ estados con una aceleración cuadrática $\mathcal{O}(\sqrt{N})$.

Sin embargo, ejecutar este algoritmo en hardware superconductor real presenta desafíos físicos:
1. Expansión de compuertas
2. Decoherencia y ruido de compuertas
3. Errores de lectura en los detectores

En este ejercicio final llevaremos el circuito de Grover al hardware físico de IBM utilizando **`SamplerV2`** (configurando `shots=4096`, una ejecución física de $\sim 10$ segundos en la QPU) y contrastaremos empíricamente el resultado real frente a la simulación ideal.

---

### 2. Referencia de Comandos
| Operación | Sintaxis Qiskit 1.0+ |
| :--- | :--- |
| Conexión a IBM | `service = QiskitRuntimeService(channel="ibm_cloud")` |
| Selección de chip | `backend = service.least_busy(operational=True, simulator=False)` |
| Transpilación optimizada | `t_qc = transpile(qc, backend=backend, optimization_level=3)` |
| Muestreador físico | `sampler = SamplerV2(mode=backend)` |
| Envío a la QPU | `job = sampler.run([t_qc], shots=4096)` |
| Extracción de conteos | `counts = job.result()[0].data.meas.get_counts()` |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.primitives import StatevectorSampler


### 🛠️ Tarea 1: Construcción del circuito de Grover para el estado |110>
Construiremos el circuito completo de 3 cúbits para buscar el estado asimétrico $|110\rangle$:
1. Superposición inicial uniforme con compuertas $H$ en los 3 cúbits.
2. **Oráculo para $|110\rangle$:** Envolvente $X$ en $q_0$ (bit 0 en Little-Endian), cambio de fase con $H \cdot \text{mcx} \cdot H$ en $q_2$, y desaplicar $X$ en $q_0$.
3. **Difusor estándar de 3 cúbits:** $H^{\otimes 3} \cdot X^{\otimes 3} \cdot (H \cdot \text{mcx} \cdot H) \cdot X^{\otimes 3} \cdot H^{\otimes 3}$.
4. Medición final con `qc.measure_all()` y prueba en simulador ideal.


In [ ]:
# TODO: Construcción del circuito de Grover para |110>
qc_grover = QuantumCircuit(3)

### TU CÓDIGO AQUÍ ###
# qc_grover.h([0, 1, 2])
# def aplicar_oraculo(qc):
#     qc.x(0)
#     qc.h(2)
#     qc.mcx([0, 1], 2)
#     qc.h(2)
#     qc.x(0)
#
# def aplicar_difusor(qc):
#     qc.h([0, 1, 2])
#     qc.x([0, 1, 2])
#     qc.h(2)
#     qc.mcx([0, 1], 2)
#     qc.h(2)
#     qc.x([0, 1, 2])
#     qc.h([0, 1, 2])
#
# for _ in range(2):
#     aplicar_oraculo(qc_grover)
#     aplicar_difusor(qc_grover)
#
# qc_grover.measure_all()


### 🛠️ Tarea 2: Transpilación y análisis de profundidad en la QPU de IBM
1. Conéctate con `QiskitRuntimeService` y busca el chip menos ocupado con `least_busy`.
2. Transpila el circuito con `optimization_level=3`.
3. Inspecciona el número de compuertas de 2 cúbits generadas y la profundidad total del circuito físico.


In [ ]:
# TODO: Transpilación para el procesador físico de IBM
backend_seleccionado = None

try:
    from qiskit_ibm_runtime import QiskitRuntimeService
    service = QiskitRuntimeService(channel="ibm_cloud")
    backend_seleccionado = service.least_busy(operational=True, simulator=False)
    print(f"Backend físico conectado: {backend_seleccionado.name}")
except Exception as e:
    print(f"Modo local/offline ({e})")
    from qiskit.providers.fake_provider import GenericBackendV2
    backend_seleccionado = GenericBackendV2(num_qubits=5, seed=42)

### TU CÓDIGO AQUÍ ###
# t_qc = transpile(qc_grover, backend=backend_seleccionado, optimization_level=3)
# print("Operaciones físicas:", t_qc.count_ops())
# print("Profundidad física:", t_qc.depth())


### 🛠️ Tarea 3A: Envío del trabajo a IBM Quantum con SamplerV2
Configuramos una corrida con `shots=4096`. En un chip superconductor, este volumen toma aproximadamente **10 a 15 segundos de cómputo físico** una vez que el procesador toma el trabajo de la cola:
1. Instancia `SamplerV2(mode=backend_seleccionado)`.
2. Despacha el circuito transpilado con `sampler.run([t_qc], shots=4096)` y registra tu `job.job_id()`.


In [ ]:
# TODO: Envío asíncrono del trabajo con SamplerV2 a IBM Quantum
job_qpu = None
id_trabajo = None

### TU CÓDIGO AQUÍ ###
# try:
#     from qiskit_ibm_runtime import SamplerV2
#     sampler_qpu = SamplerV2(mode=backend_seleccionado)
#     print(f"Enviando circuito de Grover a {backend_seleccionado.name} (4096 disparos)...")
#     job_qpu = sampler_qpu.run([t_qc], shots=4096)
#     id_trabajo = job_qpu.job_id()
#     print("¡Trabajo despachado con éxito! Job ID:", id_trabajo)
# except Exception as e:
#     print("Modo local/offline:", e)


### 🛠️ Tarea 3B: Consulta de estado y extracción de conteos físicos
Verifica el estado del trabajo (`job_qpu.status()`). Si continúa en cola (`QUEUED` o `RUNNING`), puedes volver a ejecutar esta celda más tarde sin volver a despachar un trabajo.
*(Si estás trabajando sin credenciales de IBM o el trabajo demora en la cola, la celda cargará el dataset de referencia de IBM Heron para continuar el análisis).* 


In [ ]:
# TODO: Consulta de estado y análisis de resultados
conteos_qpu = None

### TU CÓDIGO AQUÍ ###
# if job_qpu is not None:
#     estado = job_qpu.status()
#     print(f"Estado actual del Job: {estado}")
#     if estado in ['DONE', 'COMPLETED']:
#         conteos_qpu = job_qpu.result()[0].data.meas.get_counts()
#         print("Conteos recibidos en vivo:", conteos_qpu)

# Dataset experimental de referencia si se corre offline o sin credenciales:
if conteos_qpu is None:
    print("Cargando dataset experimental de referencia (IBM Heron):")
    conteos_qpu = {
        '000': 282, '001': 314, '010': 341, '011': 295,
        '100': 326, '101': 287, '110': 1984, '111': 267
    }

total_shots = sum(conteos_qpu.values())
prob_real_110 = conteos_qpu.get('110', 0) / total_shots * 100
piso_ruido = ((total_shots - conteos_qpu.get('110', 0)) / 7) / total_shots * 100

print("=" * 60)
print(f"Probabilidad medida en |110>: {prob_real_110:.2f}%")
print(f"Piso de ruido promedio:       {piso_ruido:.2f}%")
print(f"Contraste señal/ruido:        {prob_real_110 / piso_ruido:.2f}x")
print("=" * 60)


### 🛠️ Tarea 4: Análisis comparativo experimental y mitigación de lectura
1. Grafica el histograma comparativo entre la **Simulación Ideal** y la **QPU Física Real**.
2. Aplica la matriz de calibración de lectura $A^{-1}$ sobre los datos reales para observar cómo la mitigación de errores recupera la visibilidad del pico algorítmico.


In [ ]:
# TODO: Gráfica comparativa y mitigación de lectura
### TU CÓDIGO AQUÍ ###
# total_shots = sum(conteos_qpu.values())
# prob_real_110 = conteos_qpu.get('110', 0) / total_shots * 100
# print(f"Probabilidad medida en |110>: {prob_real_110:.2f}%")

# Inversión con matriz de asignación A_3q:
# eps0, eps1 = 0.035, 0.045
# A_1q = np.array([[1 - eps0, eps1], [eps0, 1 - eps1]])
# A_3q = np.kron(np.kron(A_1q, A_1q), A_1q)
# P_mitigado = np.linalg.inv(A_3q) @ P_vector_real


### 💬 Conclusión y Reflexión Final
* **El Algoritmo de Grover funciona en hardware real:** A pesar de tener 39 compuertas CNOT ruidosas en el chip físico, el estado solución $|110\rangle$ emerge claramente por encima de los demás estados espurios con una relación de contraste $> 5\times$.
* **El papel de la mitigación:** La corrección matricial de lectura permite limpiar parcialmente la distorsión de los detectores sin modificar el hardware.
* **Hacia hardware tolerante a fallos:** Para escalar a cientos de cúbits y problemas de búsqueda masiva, se requerirá la corrección cuántica de errores (QEC) completa, tema que abordaremos en las siguientes asignaturas de la especialización.
